In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hugomathien/soccer")

print("Path to dataset files:", path)

In [ ]:
import sqlite3
import pandas as pd

db_path = path + "/database.sqlite"
conn = sqlite3.connect(db_path)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

In [ ]:
match = pd.read_sql("SELECT * FROM Match;", conn)
team = pd.read_sql("SELECT * FROM Team;", conn)
league = pd.read_sql("SELECT * FROM League;", conn)
country = pd.read_sql("SELECT * FROM Country;", conn)

print(match.shape)
match.head()

In [ ]:
match.info()
match[['home_team_goal', 'away_team_goal']].describe()

In [ ]:
# Selecionar só as colunas essenciais para começar
cols = ['id', 'country_id', 'league_id', 'season', 'stage', 'date',
        'match_api_id', 'home_team_api_id', 'away_team_api_id',
        'home_team_goal', 'away_team_goal']

match_clean = match[cols].copy()

# Trazer nomes dos times (merge para casa e para fora separadamente)
team_names = team[['team_api_id', 'team_long_name']]

match_clean = match_clean.merge(
    team_names.rename(columns={'team_api_id': 'home_team_api_id', 'team_long_name': 'home_team'}),
    on='home_team_api_id', how='left'
)
match_clean = match_clean.merge(
    team_names.rename(columns={'team_api_id': 'away_team_api_id', 'team_long_name': 'away_team'}),
    on='away_team_api_id', how='left'
)

# Trazer nome do país/liga
match_clean = match_clean.merge(country.rename(columns={'id': 'country_id', 'name': 'country'}), on='country_id', how='left')

print(match_clean.shape)
match_clean[['date', 'season', 'country', 'home_team', 'away_team', 'home_team_goal', 'away_team_goal']].head(10)

In [ ]:
def get_result(row):
    if row['home_team_goal'] > row['away_team_goal']:
        return 'H'  # vitória do mandante
    elif row['home_team_goal'] < row['away_team_goal']:
        return 'A'  # vitória do visitante
    else:
        return 'D'  # empate

match_clean['result'] = match_clean.apply(get_result, axis=1)
match_clean['result'].value_counts(normalize=True)

In [ ]:
match_clean.to_sql('match_clean', conn, index=False, if_exists='replace')

In [ ]:
query1 = """
SELECT country,
       COUNT(*) as total_partidas,
       ROUND(100.0 * SUM(CASE WHEN result = 'H' THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_vitoria_casa
FROM match_clean
GROUP BY country
ORDER BY pct_vitoria_casa DESC;
"""
pd.read_sql(query1, conn)

In [ ]:
query2 = """
SELECT home_team,
       COUNT(*) as jogos_casa,
       SUM(CASE WHEN result = 'H' THEN 1 ELSE 0 END) as vitorias_casa,
       ROUND(100.0 * SUM(CASE WHEN result = 'H' THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_vitoria
FROM match_clean
GROUP BY home_team
HAVING COUNT(*) >= 50
ORDER BY pct_vitoria DESC
LIMIT 10;
"""
pd.read_sql(query2, conn)

In [ ]:
query3 = """
SELECT season,
       ROUND(AVG(home_team_goal + away_team_goal), 2) as media_gols_por_partida
FROM match_clean
GROUP BY season
ORDER BY season;
"""
pd.read_sql(query3, conn)

In [ ]:
import matplotlib.pyplot as plt

df_country = pd.read_sql(query1, conn)

plt.figure(figsize=(10,5))
plt.bar(df_country['country'], df_country['pct_vitoria_casa'], color='steelblue')
plt.xticks(rotation=45)
plt.ylabel('% Vitórias do Mandante')
plt.title('Taxa de Vitória em Casa por País (2008-2016)')
plt.tight_layout()
plt.savefig('vitoria_casa_por_pais.png', dpi=150)
plt.show()

In [ ]:
df_teams = pd.read_sql(query2, conn)

plt.figure(figsize=(10,5))
plt.barh(df_teams['home_team'], df_teams['pct_vitoria'], color='darkorange')
plt.xlabel('% Vitórias em Casa')
plt.title('Top 10 Times com Maior Taxa de Vitória em Casa (mín. 50 jogos)')
plt.gca().invert_yaxis()  # deixa o maior no topo
plt.tight_layout()
plt.savefig('top_times_vitoria_casa.png', dpi=150)
plt.show()

In [ ]:
df_season = pd.read_sql(query3, conn)

plt.figure(figsize=(10,5))
plt.plot(df_season['season'], df_season['media_gols_por_partida'], marker='o', color='seagreen')
plt.xticks(rotation=45)
plt.ylabel('Média de Gols por Partida')
plt.title('Evolução da Média de Gols por Temporada (2008-2016)')
plt.tight_layout()
plt.savefig('gols_por_temporada.png', dpi=150)
plt.show()